In [ ]:
!pip -q install transformers datasets jiwer sentencepiece accelerate peft safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from jiwer import wer, cer

from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from peft import PeftModel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Already processed ASR CSV
ASR_RESULTS_CSV = "/content/drive/MyDrive/hint_pipeline_asr_results_1000.csv"

# XLM-R dual-head model
XLMR_OUTPUT_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2"
XLMR_MODEL_DIR = os.path.join(XLMR_OUTPUT_DIR, "best_model")
ROOT_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2root.json")
SUFFIX_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2suffix.json")
TRAIN_DATA_PATH = "/content/drive/MyDrive/dual_head_training_data.csv"

# Newly trained mT5 LoRA adapter
BASE_MT5_MODEL = "google/mt5-large"
MT5_LORA_ADAPTER_DIR = "/content/drive/MyDrive/mt5_large_lora_r16_experiment_D_fixed/best_lora_adapter"

# Outputs
HINT_RESULTS_CSV = "/content/drive/MyDrive/new_pipeline_with_xlmr_hints_1000.csv"
FINAL_RESULTS_CSV = "/content/drive/MyDrive/new_pipeline_mt5_lora_results_1000.csv"
FINAL_METRICS_JSON = "/content/drive/MyDrive/new_pipeline_mt5_lora_metrics_1000.json"
DEBUG_HINTS_CSV = "/content/drive/MyDrive/new_pipeline_debug_hints_1000.csv"

MAX_LEN_XLMR = 96

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
print("USE_BF16:", USE_BF16)

Device: cuda
USE_BF16: True


In [ ]:
asr_df = pd.read_csv(ASR_RESULTS_CSV)

print("Rows:", len(asr_df))
print("Columns:", asr_df.columns.tolist())

required_cols = ["id", "audio_wav_path", "duration", "expected_text", "generated_text", "error"]

for col in required_cols:
    if col not in asr_df.columns:
        raise ValueError(f"Missing column: {col}")

asr_df["expected_text"] = asr_df["expected_text"].astype(str).str.strip()
asr_df["generated_text"] = asr_df["generated_text"].astype(str).str.strip()

asr_df.head()

Rows: 1000
Columns: ['id', 'audio_wav_path', 'duration', 'expected_text', 'generated_text', 'error']


,id,audio_wav_path,duration,expected_text,generated_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,NaN
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,NaN
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,NaN
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,NaN
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,NaN


In [ ]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok

def tokenize(text):
    return [clean_token(tok) for tok in str(text).strip().split() if clean_token(tok)]

def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", token))

def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", text))

def is_mixed_token(token):
    if "-" not in token:
        return False

    parts = token.split("-", 1)
    if len(parts) != 2:
        return False

    left, right = parts[0].strip(), parts[1].strip()
    return is_english_word(left) and is_tamil_text(right)

def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"

def split_mixed_token(token):
    if "-" not in token:
        return None, None
    left, right = token.split("-", 1)
    return left.strip(), right.strip()

def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""
    elif token_class == "EN":
        return token, "NULL"
    return "", ""

def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return f"LEFT={left_context} TOKEN={token} RIGHT={right_context} CLASS={token_class} ROOT={root} SUFFIX={suffix}"

def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx-window):idx]
    right_tokens = tokens[idx+1:idx+1+window]
    return " ".join(left_tokens).strip(), " ".join(right_tokens).strip()

In [ ]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)

with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

train_df = pd.read_csv(TRAIN_DATA_PATH)
changed_df = train_df[train_df["generated_token"] != train_df["expected_token"]].copy()

from collections import Counter

token_change_counter = Counter(changed_df["generated_token"].astype(str).tolist())

SUSPICIOUS_TOKEN_MIN_COUNT = 2

suspicious_tokens = {
    tok for tok, cnt in token_change_counter.items()
    if cnt >= SUSPICIOUS_TOKEN_MIN_COUNT
}

print("Root labels:", len(id2root))
print("Suffix labels:", len(id2suffix))
print("Suspicious tokens:", len(suspicious_tokens))

Root labels: 796
Suffix labels: 62
Suspicious tokens: 1803


In [ ]:
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

xlmr_model = XLMRDualHeadModel(
    model_name="xlm-roberta-large",
    num_root_labels=len(id2root),
    num_suffix_labels=len(id2suffix)
)

state_dict_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")

if os.path.exists(state_dict_path):
    from safetensors.torch import load_file
    state_dict = load_file(state_dict_path)
else:
    state_dict_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")
    state_dict = torch.load(state_dict_path, map_location="cpu")

xlmr_model.load_state_dict(state_dict)
xlmr_model.to(device)
xlmr_model.eval()

print("Loaded trained XLM-R dual-head model.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded trained XLM-R dual-head model.


In [ ]:
def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_XLMR,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root
    else:
        corrected_token = f"{pred_root}-{pred_suffix}"

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [ ]:
ROOT_CONF_THRESH_MIX = 0.95
SUFFIX_CONF_THRESH_MIX = 0.90

def should_correct_token(pred_info, suspicious_tokens):
    token_class = pred_info["token_class"]
    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    if token_class != "MIX":
        return False

    if corrected_token == original_token:
        return False

    wrong_root = pred_info.get("wrong_root")
    pred_root = pred_info.get("pred_root")

    # Safer rule: only suffix correction when English root is same
    if wrong_root != pred_root:
        return False

    if (
        pred_info["root_conf"] >= ROOT_CONF_THRESH_MIX and
        pred_info["suffix_conf"] >= SUFFIX_CONF_THRESH_MIX
    ):
        return True

    return False

In [ ]:
def build_hints_for_sentence(asr_sentence, suspicious_tokens):
    tokens = tokenize(asr_sentence)

    hints = []
    debug_rows = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)
        apply_change = should_correct_token(pred_info, suspicious_tokens)

        if apply_change:
            original_token = pred_info["original_token"]
            corrected_token = pred_info["corrected_token"]
            hints.append(f"{original_token}=>{corrected_token}")

        debug_rows.append({
            "index": idx,
            "original_token": pred_info["original_token"],
            "token_class": pred_info["token_class"],
            "wrong_root": pred_info.get("wrong_root"),
            "wrong_suffix": pred_info.get("wrong_suffix"),
            "pred_root": pred_info.get("pred_root"),
            "pred_suffix": pred_info.get("pred_suffix"),
            "root_conf": pred_info.get("root_conf"),
            "suffix_conf": pred_info.get("suffix_conf"),
            "corrected_token": pred_info.get("corrected_token"),
            "apply_hint": apply_change
        })

    hint_text = " ; ".join(hints)

    if hint_text.strip() == "":
        hint_text = "Null"

    return hint_text, debug_rows

In [ ]:
hint_rows = []
all_debug_rows = []

for idx, row in tqdm(asr_df.iterrows(), total=len(asr_df), desc="Building XLM-R hints"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        hint_text = "Null"
        debug_rows = []
    else:
        hint_text, debug_rows = build_hints_for_sentence(
            generated_text,
            suspicious_tokens
        )

    hint_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "hint_text": hint_text,
        "error": error_msg
    })

    for d in debug_rows:
        d["row_id"] = row_id
        d["generated_text"] = generated_text
        all_debug_rows.append(d)

hint_df = pd.DataFrame(hint_rows)
hint_df.to_csv(HINT_RESULTS_CSV, index=False, encoding="utf-8-sig")

debug_df = pd.DataFrame(all_debug_rows)
debug_df.to_csv(DEBUG_HINTS_CSV, index=False, encoding="utf-8-sig")

print("Saved hint CSV:", HINT_RESULTS_CSV)
print("Saved debug CSV:", DEBUG_HINTS_CSV)

hint_df["has_hint"] = hint_df["hint_text"].astype(str).str.strip() != "Null"

print("Rows with hints:", hint_df["has_hint"].sum())
print("Hint rate:", hint_df["has_hint"].mean())

hint_df.head()

Building XLM-R hints:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved hint CSV: /content/drive/MyDrive/new_pipeline_with_xlmr_hints_1000.csv
Saved debug CSV: /content/drive/MyDrive/new_pipeline_debug_hints_1000.csv
Rows with hints: 78
Hint rate: 0.078


,id,audio_wav_path,duration,expected_text,generated_text,hint_text,error,has_hint
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Null,NaN,False
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,Null,NaN,False
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,Null,NaN,False
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Null,NaN,False
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,Null,NaN,False


In [ ]:
!pip uninstall torchao

In [ ]:
mt5_tokenizer = AutoTokenizer.from_pretrained(BASE_MT5_MODEL)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MT5_MODEL,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32
)

mt5_model = PeftModel.from_pretrained(
    base_model,
    MT5_LORA_ADAPTER_DIR
)

mt5_model.to(device)
mt5_model.eval()

print("Loaded newly trained mT5 LoRA adapter.")

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded newly trained mT5 LoRA adapter.


In [ ]:
def build_mt5_prompt(asr_text, hint_text):
    asr_text = str(asr_text).strip()
    hint_text = str(hint_text).strip()

    if hint_text == "" or hint_text.lower() == "nan":
        hint_text = "Null"

    prompt = (
        "fix ASR with hints:\n"
        f"ASR: {asr_text}\n"
        f"HINT: {hint_text}"
    )

    return prompt


def run_mt5_inference_with_hints(asr_text, hint_text, max_input_len=256, max_new_tokens=128):
    try:
        prompt = build_mt5_prompt(asr_text, hint_text)

        inputs = mt5_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_len
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            output_ids = mt5_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                num_beams=1,
                do_sample=False
            )

        prediction = mt5_tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        return prediction, None

    except Exception as e:
        return "", str(e)

In [ ]:
test_row = hint_df.iloc[0]

print("ASR:")
print(test_row["generated_text"])

print("\nHint:")
print(test_row["hint_text"])

pred, err = run_mt5_inference_with_hints(
    test_row["generated_text"],
    test_row["hint_text"]
)

print("\nPrediction:")
print(pred)

print("\nReference:")
print(test_row["expected_text"])

print("\nError:")
print(err)

ASR:
Online class join பண்ண late ஆனதால start miss ஆயிட்டு

Hint:
Null

Prediction:
Online class join பண்ண late ஆனதால start miss ஆயிட்டு

Reference:
Online class join பண்ண late ஆனதால start miss ஆயிட்டு

Error:
None


In [ ]:
#new
def is_low_quality_asr(text):
    text = str(text)

    # too short
    if len(text.split()) <= 2:
        return False

    # many English tokens → likely error in Tamil context
    tokens = text.split()
    en_count = sum(1 for t in tokens if re.match(r"[A-Za-z]+", t))

    if en_count >= 3:
        return True

    return False

In [ ]:
#new

def run_mt5_with_validation(asr_text, hint_text):
    pred, err = run_mt5_inference_with_hints(asr_text, hint_text)

    if err is not None:
        return asr_text, err

    # 🔥 decision: accept only if change is meaningful
    if pred.strip() == "":
        return asr_text, None

    # simple heuristic: if output too different → reject
    diff_ratio = wer([asr_text], [pred])

    if diff_ratio > 0.6:
        return asr_text, None

    return pred, None

In [ ]:
#modfied 2

final_rows = []

for idx, row in tqdm(hint_df.iterrows(), total=len(hint_df), desc="Running mT5 with confidence gate"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    hint_text = str(row["hint_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    mt5_avg_conf = None
    mt5_min_conf = None
    used_mt5 = False
    accepted_mt5 = False

    if has_error:
        mt5_text = ""
        mt5_error = error_msg

    else:
        if hint_text == "" or hint_text == "Null":
            mt5_text = generated_text
            mt5_error = None

        else:
            used_mt5 = True

            pred_text, avg_conf, min_conf, mt5_error = run_mt5_inference_with_score(
                generated_text,
                hint_text
            )

            mt5_avg_conf = avg_conf
            mt5_min_conf = min_conf

            if mt5_error is not None:
                mt5_text = generated_text
            else:
                accepted_mt5 = accept_mt5_correction(
                    generated_text,
                    pred_text,
                    avg_conf,
                    min_conf
                )

                if accepted_mt5:
                    mt5_text = pred_text
                else:
                    mt5_text = generated_text

    final_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "hint_text": hint_text,
        "mt5_with_hints_text": mt5_text,
        "used_mt5": used_mt5,
        "accepted_mt5": accepted_mt5,
        "mt5_avg_conf": mt5_avg_conf,
        "mt5_min_conf": mt5_min_conf,
        "error": mt5_error
    })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(FINAL_RESULTS_CSV, index=False, encoding="utf-8-sig")

final_df.head()

Running mT5 with confidence gate:   0%|          | 0/1000 [00:00<?, ?it/s]

,id,audio_wav_path,duration,expected_text,generated_text,hint_text,mt5_with_hints_text,used_mt5,accepted_mt5,mt5_avg_conf,mt5_min_conf,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Null,Online class join பண்ண late ஆனதால start miss ஆ...,False,False,NaN,NaN,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,Null,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,False,False,NaN,NaN,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,Null,Itsy help center site-ல selling on itsy-ல paym...,False,False,NaN,NaN,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Null,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,False,False,NaN,NaN,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,Null,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,False,False,NaN,NaN,None


In [ ]:
print("Rows with hints:", (final_df["hint_text"] != "Null").sum())
print("Rows used mT5:", final_df["used_mt5"].sum())
print("Rows accepted mT5:", final_df["accepted_mt5"].sum())

Rows with hints: 78
Rows used mT5: 78
Rows accepted mT5: 77


In [ ]:
final_rows = []

for idx, row in tqdm(hint_df.iterrows(), total=len(hint_df), desc="Running newly trained mT5"):
    row_id = row["id"]
    audio_path = row["audio_wav_path"]
    duration = row["duration"]
    expected_text = str(row["expected_text"]).strip()
    generated_text = str(row["generated_text"]).strip()
    hint_text = str(row["hint_text"]).strip()
    error_msg = row["error"]

    has_error = (not pd.isna(error_msg)) and (str(error_msg).strip() != "")

    if has_error:
        mt5_text = ""
        mt5_error = error_msg

    else:
      # 🔥 NEW FILTER
      low_quality = is_low_quality_asr(generated_text)

      if hint_text == "Null":
          mt5_text = generated_text
          mt5_error = None

      elif not low_quality:
          # skip good ASR sentences
          mt5_text = generated_text
          mt5_error = None

      else:
          mt5_text, mt5_error = run_mt5_inference_with_hints(
              generated_text,
              hint_text
          )

    final_rows.append({
        "id": row_id,
        "audio_wav_path": audio_path,
        "duration": duration,
        "expected_text": expected_text,
        "generated_text": generated_text,
        "hint_text": hint_text,
        "mt5_with_hints_text": mt5_text,
        "error": mt5_error
    })

final_df = pd.DataFrame(final_rows)
final_df.to_csv(FINAL_RESULTS_CSV, index=False, encoding="utf-8-sig")

print("Saved final results:", FINAL_RESULTS_CSV)
final_df.head()

Running newly trained mT5:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved final results: /content/drive/MyDrive/new_pipeline_mt5_lora_results_1000.csv


,id,audio_wav_path,duration,expected_text,generated_text,hint_text,mt5_with_hints_text,error
0,8a13434a-3693-40d9-986a-a231316ecda2,/content/drive/MyDrive/audio_wav_final/audio-r...,4.62,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...,Null,Online class join பண்ண late ஆனதால start miss ஆ...,None
1,dc8ef55c-1fa0-4d9b-961f-7433693106f4,/content/drive/MyDrive/audio_wav_final/audio-r...,4.14,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,Null,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,None
2,fd532190-fb3d-4bce-9139-c1cc6a644601,/content/drive/MyDrive/audio_wav_final/audio-r...,11.64,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...,Null,Itsy help center site-ல selling on itsy-ல paym...,None
3,d569cdd8-29e6-44fc-9a4a-25a756d18ff8,/content/drive/MyDrive/audio_wav_final/audio-r...,5.28,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Null,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,None
4,1fb0ffeb-1183-4e84-8c64-eadffb685a89,/content/drive/MyDrive/audio_wav_final/audio-r...,4.86,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,Null,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,None


In [ ]:
valid_final_df = final_df[
    final_df["error"].isna() | (final_df["error"].astype(str).str.strip() == "")
].copy()

expected_list = valid_final_df["expected_text"].astype(str).str.strip().tolist()
asr_list = valid_final_df["generated_text"].astype(str).str.strip().tolist()
mt5_list = valid_final_df["mt5_with_hints_text"].astype(str).str.strip().tolist()

metrics = {
    "num_rows_total": int(len(final_df)),
    "num_rows_valid": int(len(valid_final_df)),

    "asr_wer": float(wer(expected_list, asr_list)),
    "asr_cer": float(cer(expected_list, asr_list)),

    "xlmr_hint_mt5_wer": float(wer(expected_list, mt5_list)),
    "xlmr_hint_mt5_cer": float(cer(expected_list, mt5_list)),
}

print(json.dumps(metrics, indent=2, ensure_ascii=False))

with open(FINAL_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved metrics:", FINAL_METRICS_JSON)

{
  "num_rows_total": 1000,
  "num_rows_valid": 1000,
  "asr_wer": 0.1361609832509941,
  "asr_cer": 0.041790079324549756,
  "xlmr_hint_mt5_wer": 0.140619351729124,
  "xlmr_hint_mt5_cer": 0.0460456882170701
}
Saved metrics: /content/drive/MyDrive/new_pipeline_mt5_lora_metrics_1000.json


In [ ]:
final_df[[
    "generated_text",
    "hint_text",
    "expected_text",
    "mt5_with_hints_text"
]].head(30)

,generated_text,hint_text,expected_text,mt5_with_hints_text
0,Online class join பண்ண late ஆனதால start miss ஆ...,Null,Online class join பண்ண late ஆனதால start miss ஆ...,Online class join பண்ண late ஆனதால start miss ஆ...
1,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,Null,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...,ரெண்டு table book பண்ணுங்க please ஜன்னல் பக்கம...
2,Itsy help center site-ல selling on itsy-ல paym...,Null,Etsy help center site-ல selling on etsy-ல paym...,Itsy help center site-ல selling on itsy-ல paym...
3,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Null,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...,Government-ஆல ஒரு scholarship குடுக்கினம் ஒருக...
4,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,Null,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...,நீங்க இப்பிடி சாப்பிட்டா sugar வரும் பாத்து சா...
5,Tubelight சரியா எரியேல ஒருக்கா repair செய்யணும்,Null,Tubelight சரியா எரியேல ஒருக்கா repair செய்யணும்,Tubelight சரியா எரியேல ஒருக்கா repair செய்யணும்
6,பிளவுட்ட questions-அ அவர்கிட்ட கேட்டா explain ...,Null,பிழை விட்ட questions-அ அவர்ட்ட கேட்டா explain ...,பிளவுட்ட questions-அ அவர்கிட்ட கேட்டா explain ...
7,என்ர cycle puncture ஆயிட்டு அதை repair-க்கு கொ...,Null,என்ர cycle puncture ஆயிட்டு அதான் repair-க்கு ...,என்ர cycle puncture ஆயிட்டு அதை repair-க்கு கொ...
8,Office system login problem இருந்ததால it suppo...,Null,Office system login problem இருந்ததால it suppo...,Office system login problem இருந்ததால it suppo...
9,இந்த chocolate செம்ம taste-ஆ இருக்கும் வாங்குவம்,Null,இந்த chocolate செம்ம taste-ஆ இருக்கும் வாங்குவம்,இந்த chocolate செம்ம taste-ஆ இருக்கும் வாங்குவம்


In [ ]:
hint_only_df = final_df[final_df["hint_text"].astype(str).str.strip() != "Null"].copy()

print("Rows with hints:", len(hint_only_df))

expected_hint = hint_only_df["expected_text"].astype(str).str.strip().tolist()
asr_hint = hint_only_df["generated_text"].astype(str).str.strip().tolist()
mt5_hint = hint_only_df["mt5_with_hints_text"].astype(str).str.strip().tolist()

print("ASR WER on hint rows:", wer(expected_hint, asr_hint))
print("Pipeline WER on hint rows:", wer(expected_hint, mt5_hint))

print("ASR CER on hint rows:", cer(expected_hint, asr_hint))
print("Pipeline CER on hint rows:", cer(expected_hint, mt5_hint))

Rows with hints: 12
ASR WER on hint rows: 0.21505376344086022
Pipeline WER on hint rows: 0.08602150537634409
ASR CER on hint rows: 0.04606240713224369
Pipeline CER on hint rows: 0.031203566121842496


In [ ]:
# Run only on rows where ASR is wrong
final_df["row_wer"] = [
    wer([ref], [hyp])
    for ref, hyp in zip(
        final_df["expected_text"].astype(str),
        final_df["generated_text"].astype(str)
    )
]

wrong_rows_df = final_df[final_df["row_wer"] > 0].copy()

print("Wrong ASR rows:", len(wrong_rows_df))

Wrong ASR rows: 551


In [ ]:
print("ASR WER (hint rows):", wer(expected_hint, asr_hint))
print("Pipeline WER (hint rows):", wer(expected_hint, mt5_hint))

ASR WER (hint rows): 0.21505376344086022
Pipeline WER (hint rows): 0.08602150537634409
